In [ ]:
import os
import time

import pandas as pd

# External Python files
from stage_1.facade_extraction import FacadeExtractor
from stage_1.src.api_request import get_pano_id

# API options
BBOX_REGION = [122300.00, 486628.78, 122400.00, 486475.48]
MISSION_YEAR = 2025  # API tag
SEARCH_RADIUS = 10  # Building search radius from an initial panoramic image location

# Texture options
TEXTURE_RESOLUTION = 30
IMAGE_HEIGHT_IRL = 30  # In meters, buildings above 30m are therefore partly cropped
CAMERA_HEIGHT = 2  # Height of the camera mounted on the car

# Validation options before rectification (strict settings)
MIN_DISTANCE = 1.7  # Min distance camera to building
MAX_DISTANCE = 9
MIN_BUILDING_AREA = 15
MAX_BUILDING_AREA = 400

IMAGES_OUTPUT = "texture_output/"
OUTPUT_NAME = "output_stage1"

In [ ]:
# Create a directory, first check if it already exists
if not os.path.exists(IMAGES_OUTPUT):
    os.makedirs(IMAGES_OUTPUT)
else:
    print("The 'texture_output' folder already exists.")

print("--- Start ---")

# Start timer
start_time = time.time()

# Get the panoramas (from API)
pano_ids = get_pano_id(MISSION_YEAR, BBOX_REGION)

print(f"Found {len(pano_ids)} panorama images")

In [ ]:
pano_ids

In [ ]:
print("Step 1: Identify visible building edges in range of a panoramic image")

facade_extractor = FacadeExtractor(
    texture_output_folder=IMAGES_OUTPUT,
    camera_height=CAMERA_HEIGHT,
    image_height_irl=IMAGE_HEIGHT_IRL,
    texture_resolution=TEXTURE_RESOLUTION,
    min_building_area=MIN_BUILDING_AREA,
    max_building_area=MAX_BUILDING_AREA,
    min_building_distance=MIN_DISTANCE,
    max_building_distance=MAX_DISTANCE,
    building_search_radius=SEARCH_RADIUS,
)

In [ ]:
from stage_1.src import api_request

pano_id = pano_ids[45]

facade_data = list()

# Get the panoramic image url and the location of the "observer"
image_url, observer, observer_wgs = api_request.get_pano_image(pano_id)

# Get BAG "pand" information. Search radius in meters from point "near"
pand_json = api_request.get_buildings_in_range(
    observer_wgs, SEARCH_RADIUS
)

# Get a dict of the buildings (footprint data) in range
building_polygons = api_request.get_building_polygons(pand_json)

# Validation checks on the GPS location and building polygons
building_polygons = facade_extractor.get_admissible_data(observer, building_polygons)

if not building_polygons:
    print("No admissible building polygons in range. Go to the next panorama.")
else:
    print(f"{len(building_polygons)} buildings in range.")

In [ ]:
from stage_1.src import visible_view

# Generate the outlines of the arrangement
arr = visible_view.generate_arrangement(
    observer, building_polygons.values()
)

# Get the Visibility Polygon for point q
visibility_polygon = visible_view.get_visibility_polygon(arr, observer)

# Get two most distant visible points of a builing
facade_extractor.get_facade_coordinates(
    building_polygons, visibility_polygon, observer, image_url, facade_data
)

In [ ]:
import shapely.geometry as sg
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10,8))

ax.plot(*observer, marker="*", color="red")

for building in building_polygons.values():
    ax.plot(*sg.Polygon(building).exterior.xy, color="blue")

ax.plot(*visibility_polygon.exterior.xy, color="green")

for facade in facade_data:
    ax.plot(*facade["visible_point_one"], marker="*", color="purple")
    ax.plot(*facade["visible_point_two"], marker="*", color="purple")

plt.show()

In [ ]:
facade_data

In [ ]:
print("Step 1: Identify visible building edges in range of a panoramic image")

facade_extractor = FacadeExtractor(
    texture_output_folder=IMAGES_OUTPUT,
    camera_height=CAMERA_HEIGHT,
    image_height_irl=IMAGE_HEIGHT_IRL,
    texture_resolution=TEXTURE_RESOLUTION,
    min_building_area=MIN_BUILDING_AREA,
    max_building_area=MAX_BUILDING_AREA,
    min_building_distance=MIN_DISTANCE,
    max_building_distance=MAX_DISTANCE,
    building_search_radius=SEARCH_RADIUS,
)

facade_data = list()

facade_extractor.get_visible_building_edges(pano_ids, facade_data)

# Create pandas dataframe to easily manipulate the data
df = pd.DataFrame(facade_data)

In [ ]:
df["pand_id"].value_counts()

In [ ]:
df[df["pand_id"]=="0363100012170493"]

In [ ]:
building_polygons["0363100012170493"]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10,8))

colors = ["red", "orange", "gray", "green", "blue", "purple"]

ax.plot(*building_polygons["0363100012170493"].exterior.coords)

for row in df[df["pand_id"]=="0363100012170493"].reset_index(drop=True).itertuples():
    ax.plot(*row.observer, marker="*", color=colors[row.Index])
    point_one = row.visible_point_one
    point_two = row.visible_point_two
    ax.plot([point_one[0], point_two[0]], [point_one[1], point_two[1]], color=colors[row.Index])

plt.show()

In [ ]:
print("Step 2: Drop duplicate facade textures")

# Drop duplicate facade rows, keep the ones with shortest distance.
df = (
    df.sort_values("distance", ascending=True)
    .drop_duplicates(subset=["visible_point_one", "visible_point_two"])
    .sort_index()
    .reset_index(drop=True)
)


In [ ]:

print("Step 3: Extract facade textures from panoramic images")

# Group by image_url and later iterate over dataframe grouped by image_url
facade_extractor.extract_facade_texture(df)